Generate a random list of nodes and edges, specifying the strengths distributions

In [8]:
# auto-reload the packages at every run
%load_ext autoreload
%autoreload 2

#display all the results not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import numpy as np
import numpy.random as npr

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
def sample_strengths(N, stre_out_method = 'correlated'):
    """
    Sample strengths from a log-normal with fixed mu and scale
    if stre_out_method = 'correlated' a positive correlation will be set among the prop_out, prop_in
    """

    # sample the strengths in from the log-normal distribution
    mu_in, scale_in = 9.145050968796784, 3.0501786627011396
    npr.seed(0)
    logn_sample = lambda mu, scale: npr.lognormal(mean=mu, sigma=scale, size=N)
    prop_in = logn_sample(mu_in, scale_in)

    # sample the strengths out from the log-normal distribution
    mu_out, scale_out = 9.753643239750687, 3.2045983690423174
    if stre_out_method == 'uncorrelated':
        prop_out = logn_sample(mu_out, scale_out)
    elif stre_out_method == 'correlated':
        mu_eps, scale_eps = mu_out - mu_in, np.sqrt(scale_out**2 - scale_in**2)
        eps = logn_sample(mu_eps, scale_eps)
        prop_out = prop_in * eps


    return prop_out, prop_in

In [10]:
N = int(1e5)
E = 300 #int(2.3 * 1e6)

density = E / (N*(N-1))
print(f'-density: {density}',)

-density: 3.000030000300003e-08


In [11]:
prop_out, prop_in = sample_strengths(N, stre_out_method = 'correlated')

In [12]:
# # plot the distributions of the prop_in and prop_out
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 5))
# plt.hist(np.log(prop_in), bins=100, alpha=0.5, label='prop_in', density=True)
# plt.hist(np.log(prop_out), bins=100, alpha=0.5, label='prop_out', density=True)
# plt.xlabel('Strength')
# plt.ylabel('Density')
# plt.legend()
# plt.title('Distribution of Strengths')

In [13]:
from graph_ensembles import sparse as sp
import graph_ensembles.utils as utils

In [18]:
#v = np.arange(len(prop_out), dtype=np.int64)
param = 5e-16 #1.8996372e-17
kwargs_model = {
                "num_vertices" : N,
                "prop_out" : prop_out,
                "prop_in" : prop_in,
                "param" : param,
                "selfloops" : False,
                "name" : 'Invariant',
                "level" : 0,
                "vsplit" : 0,
                "intra_size" : 1,
                "fit_method" : 'num_edges_intra',
                "corpkey" : False,
                "test_graph" : "intra",
                } #git hub repositories
model = sp.ScaleInvariantModel(**kwargs_model)

import os
model.vars_dir_ensembles = f"{os.path.expanduser('~')}/Documents/Datasets/ING-Directed/xgrid_20240427_xtrans_20240424"

model.send_variables_to_gpu(arr = np.ones(1))
gs = model.sample()

tensor([True], device='cuda:0')

In [19]:
Ns = np.unique(np.concatenate(gs.adj.nonzero())).size
Es = gs.num_edges()
rho_s = Es / (Ns * (Ns - 1))

print(f'-Ns, Es, rho_s: {Ns, Es, rho_s}',)

-Ns, Es, rho_s: (66146, np.int64(3938370), np.float64(0.0009001521171299547))


Create a Dataset with naics codes similarly to ING one

In [20]:
# full naics codes
low, high = 111150, 814110
naics_codes = np.arange(low, high + 1)

# filter out the 52, 55, 72, 99 as done in the real net
permitted_naics = np.array(naics_codes // int(1e4), dtype = int)
mask = np.isin(permitted_naics, [52, 55, 72, 99])

# create a mask
naics_codes = naics_codes[~mask]

In [21]:
# set the total number of sectors
num_naics = 972
diff_naics = high - low

assert N > num_naics, "The number of sectors (num_naics) should be less than the number of nodes (N)"

# select num_naics from filtered naics_codes, then select N of them (with replacement) to create groups
np.random.seed(0)
num_naics_codes = np.random.choice(a = naics_codes, size = num_naics, replace = False)
id_naics = np.random.choice(a = num_naics_codes, size = N, replace = True)

In [22]:
import pandas as pd

# initialize the pandas DF with rows and cols
rows, cols = gs.adj.nonzero()
pdtrans = pd.DataFrame({"payer_grid_id" : rows, "beneficiary_grid_id" : cols})

def create_naics_col(df, pay_ben, id_naics):
    df[f"{pay_ben}_naics_code"] = df[f"{pay_ben}_grid_id"].map(lambda i: id_naics[i])
    df[f"{pay_ben}_naics_desc"] = pay_ben[0]
    return df

# create payer / beneficiaries naics columns, nrofpayments
pdtrans = create_naics_col(pdtrans, "payer", id_naics)
pdtrans = create_naics_col(pdtrans, "beneficiary", id_naics)
pdtrans.loc[:, "nrofpayments"] = 1

# assign the weights with the MaxEnt rule
W = np.sum(prop_out)
pdtrans.loc[:, "amount_euro"] = pdtrans.apply(lambda row: prop_out[row["payer_grid_id"]] * prop_in[row["beneficiary_grid_id"]], axis=1)
pdtrans.loc[:, "amount_euro"] /= W

# reorder the pd.DataFrame
pdtrans = pdtrans.loc[:, ["payer_grid_id", "payer_naics_code", "payer_naics_desc", "beneficiary_grid_id", "beneficiary_naics_code", "beneficiary_naics_desc", "nrofpayments", "amount_euro"]]

In [23]:
import os
dataset_folder = f"{os.path.expanduser('~')}/Documents/Datasets/ING-Directed/xgrid_20240427_xtrans_20240424"
full_path = f"{dataset_folder}/pdtrans_no_rotw_gridSelfLoops_52559299_grid_id.csv"
os.makedirs(dataset_folder, exist_ok = True)

if True: #not os.path.exists(full_path):
    pdtrans.to_csv(full_path, index = False)

### Miscellanea

Without parallelization copied from the ensembles

In [ ]:
from numba import njit, prange
from numba.typed import List
@njit()  # pragma: no cover
def _binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    """Sample from the ensemble."""
    rows = List()
    cols = List()

    N = len(prop_out)
    np.random.seed(1)
    for i in prange(N):
        p_out_i = prop_out[i]
        for j in range(N):
            if (i != j):
                p_in_j = prop_in[j]
                p = p_ij(param, p_out_i, p_in_j, prop_dyad(i, j))
                if np.random.random() < p:
                    rows.append(i)
                    cols.append(j)

        # if i % 10 == 0: print(f'-i: {i}',)
        
    return rows, cols

@njit()  # pragma: no cover
def leo_binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    """Sample from the ensemble."""
    rows = []
    cols = []
    np.random.seed(1)
    for i, p_out_i in enumerate(prop_out):
        for j, p_in_j in enumerate(prop_in):
            if (i != j) | selfloops:
                p = p_ij(param, p_out_i, p_in_j, prop_dyad(i, j))
                if np.random.random() < p:
                    rows.append(i)
                    cols.append(j)

    return rows, cols

In [ ]:
rows_fail, cols_fail = leo_binary_sample(
    model.p_ij,
    model.param,
    model.prop_out,
    model.prop_in,
    model.prop_dyad,
    model.selfloops,
)
# ipython_pygments_lexers

In [ ]:
# # sort rows, cols first wrt rows and then to cols
# idx = np.lexsort((cols_fail, rows_fail))
# rows_fail = np.array(rows_fail)[idx]
# cols_fail = np.array(cols_fail)[idx]

# idx = np.lexsort((cols, rows))
# rows = rows[idx]
# cols = cols[idx]

# if np.all(rows == rows_fail) and np.all(cols == cols_fail):
#     print(f'-rows and cols: {True}',)
# else:
#     if np.all(rows == rows_fail):
#         print(f'-rows: {True}',)
#     else:
#         rows == rows_fail

#     if np.all(cols == cols_fail):
#         print(f'-cols: {True}',)
#     else:
#         cols == cols_fail

#     rows_fail
#     rows

#     cols_fail
#     cols

-rows and cols: True
